<a href="https://colab.research.google.com/github/shafinnahian/teen-mental-health-predictive-model/blob/main/notebooks/digital_wellbeing_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Digital wellbeing classifier

Run this notebook top to bottom on a fresh Colab runtime.

We predict `digital_wellbeing_flag` (Healthy, Moderate, At Risk) from nine lifestyle variables. Stress, anxiety, addiction, and derived risk scores are excluded from the feature set.

After cloning the repo, the dataset is at:

`/content/teen-mental-health-predictive-model/data/Teen_Mental_Health.csv`

Dataset: [Kaggle — Teen Mental Health (argonnxx)](https://www.kaggle.com/datasets/argonnxx/teen-mental-health)

**Part 1 — Setup:** paths, dependencies, load CSV, verify locked config.

**Part 2 — EDA:** leakage check, class balance, lifestyle figures saved to `outputs/figures/`.

**Part 3 — Preprocessing:** stratified split, encoding, scaling, one engineered feature, saved artifacts.

**Part 4 — Run summary:** saves key results to `outputs/run_summary.zip` for download.

Course project only. Not for clinical use.


In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/teen-mental-health-predictive-model")
DATA_CSV = REPO / "data" / "Teen_Mental_Health.csv"

if not REPO.is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/shafinnahian/teen-mental-health-predictive-model.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

print("Repo:", REPO)
print("Data CSV:", DATA_CSV)
assert DATA_CSV.is_file(), f"Missing dataset at {DATA_CSV}"


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Optional

REPO = Path("/content/teen-mental-health-predictive-model")
DATA_CSV = REPO / "data" / "Teen_Mental_Health.csv"

if REPO.is_dir():
    os.chdir(REPO)


def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src" / "config.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Clone the repo into /content/teen-mental-health-predictive-model first."
    )


PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)
print("Data CSV:", DATA_CSV)

%pip install -q -r {PROJECT_ROOT / "requirements.txt"}


In [ ]:
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import config
import paths
import data_loader
import io_utils

print("Imports OK:", [config.__name__, paths.__name__, data_loader.__name__, io_utils.__name__])
print("src on sys.path:", SRC_PATH)

In [ ]:
from paths import (
    ensure_output_dirs,
    data_csv_path,
    figures_dir,
    models_dir,
    metrics_dir,
    get_project_root,
)

ensure_output_dirs()
print("PROJECT_ROOT (paths.py):", get_project_root())
print("data CSV:", data_csv_path())
print("figures:", figures_dir())
print("models:", models_dir())
print("metrics:", metrics_dir())

In [ ]:
from data_loader import load_raw_data, validate_data

df = load_raw_data()
validation = validate_data(df)
display(df.head())
validation

In [ ]:
from config import (
    TARGET,
    TASK,
    TARGET_CLASSES,
    LIFESTYLE_FEATURES,
    EXCLUDED_COLUMNS,
    SEED,
    TEST_SIZE,
    DATASET_URL,
    EXPECTED_ROWS,
    EXPECTED_COLS,
)

print("TARGET:", TARGET)
print("TASK:", TASK)
print("TARGET_CLASSES:", TARGET_CLASSES)
print("SEED:", SEED)
print("TEST_SIZE:", TEST_SIZE)
print("Expected shape:", (EXPECTED_ROWS, EXPECTED_COLS))
print("Dataset URL:", DATASET_URL)
print("Lifestyle features (" + str(len(LIFESTYLE_FEATURES)) + "):")
for col in LIFESTYLE_FEATURES:
    print(" -", col)
print("Excluded columns:")
for col in EXCLUDED_COLUMNS:
    print(" -", col)

In [ ]:
assert config.TARGET == "digital_wellbeing_flag"
assert df.shape == (1200, 16)
assert validation["missing"] == 0
assert figures_dir().is_dir()
assert models_dir().is_dir()
assert metrics_dir().is_dir()

print("Phase 1 smoke tests passed.")
print("Target counts:", validation["target_counts"])

## Exploratory data analysis

The CSV is already loaded above. This section checks for target leakage, plots class balance and lifestyle patterns, and saves figures to `outputs/figures/`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from config import TARGET, TARGET_CLASSES, EXCLUDED_COLUMNS, SEED

LIFESTYLE_NUMERIC = [
    "age",
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
]

FIGURES_DIR = figures_dir()
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)

print("FIGURES_DIR:", FIGURES_DIR)
print("Target counts:", validation["target_counts"])

## Leakage check

On every row, `mental_health_risk_score` equals `stress_level + anxiety_level + addiction_level`.

Those psychological columns stay out of the model. The table below shows their mean values by wellbeing class for documentation only.


In [ ]:
risk_equals_psycho_sum = (
    df["stress_level"] + df["anxiety_level"] + df["addiction_level"]
    == df["mental_health_risk_score"]
).all()

assert risk_equals_psycho_sum, (
    "Expected mental_health_risk_score == stress + anxiety + addiction for all rows."
)
print(
    "Leakage check PASSED: risk_score = stress + anxiety + addiction (all",
    len(df),
    "rows)",
)
print("Excluded predictors (not used for modeling):", EXCLUDED_COLUMNS)

psycho_by_wellbeing = (
    df.groupby(TARGET)[
        [
            "stress_level",
            "anxiety_level",
            "addiction_level",
            "mental_health_risk_score",
        ]
    ]
    .mean()
    .reindex(list(TARGET_CLASSES))
    .round(2)
)
display(psycho_by_wellbeing)

## Figures

Four plots, saved to `outputs/figures/`. Wellbeing classes are ordered Healthy, Moderate, At Risk.


In [ ]:
# Figure 1 — Class balance
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(
    data=df,
    x=TARGET,
    order=list(TARGET_CLASSES),
    hue=TARGET,
    hue_order=list(TARGET_CLASSES),
    palette="Set2",
    legend=False,
    ax=ax,
)
ax.set_title("Class balance: digital_wellbeing_flag")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")
plt.tight_layout()
out1 = FIGURES_DIR / "01_class_balance.png"
fig.savefig(out1, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out1)

In [ ]:
# Group means for daily social media hours (summary evidence)
social_media_stats = (
    df.groupby(TARGET)["daily_social_media_hours"]
    .agg(["mean", "median", "count"])
    .reindex(list(TARGET_CLASSES))
    .round(2)
)
display(social_media_stats)

In [ ]:
# Figure 2 — Social media hours by wellbeing class (boxplot)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(
    data=df,
    x=TARGET,
    y="daily_social_media_hours",
    order=list(TARGET_CLASSES),
    hue=TARGET,
    hue_order=list(TARGET_CLASSES),
    palette="Set2",
    legend=False,
    ax=ax,
)
ax.set_title("Daily social media hours by digital wellbeing class")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Daily social media hours")
plt.tight_layout()
out2 = FIGURES_DIR / "02_social_media_by_wellbeing.png"
fig.savefig(out2, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out2)

In [ ]:
# Figure 3 — Correlation heatmap (lifestyle numerics only; no psycho scales)
corr = df[LIFESTYLE_NUMERIC].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    ax=ax,
)
ax.set_title("Correlation heatmap — lifestyle numeric predictors")
plt.tight_layout()
out3 = FIGURES_DIR / "03_lifestyle_correlation.png"
fig.savefig(out3, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out3)
print("Columns in heatmap:", LIFESTYLE_NUMERIC)

In [ ]:
# Figure 4 — Platform usage × wellbeing (row-normalized %)
ct_counts = pd.crosstab(df["platform_usage"], df[TARGET])
ct_counts = ct_counts.reindex(columns=list(TARGET_CLASSES))
ct_pct = ct_counts.div(ct_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(
    ct_pct,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("Platform usage × wellbeing (row % within platform)")
ax.set_xlabel("Wellbeing class")
ax.set_ylabel("Platform usage")
plt.tight_layout()
out4 = FIGURES_DIR / "04_platform_crosstab.png"
fig.savefig(out4, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out4)
print("Raw counts:")
display(ct_counts)

In [ ]:
# Smoke check: all four figures written
expected_figs = [
    FIGURES_DIR / "01_class_balance.png",
    FIGURES_DIR / "02_social_media_by_wellbeing.png",
    FIGURES_DIR / "03_lifestyle_correlation.png",
    FIGURES_DIR / "04_platform_crosstab.png",
]
for path in expected_figs:
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty figure: {path}"
print("Phase 2 figure smoke tests passed.")
for path in expected_figs:
    print(" -", path)

## Summary

The file has 1,200 rows, 16 columns, and no missing values. Class counts match the locked Kaggle snapshot: Moderate 743, Healthy 306, At Risk 151. Moderate is the majority class, so accuracy alone would be a weak metric; macro-F1 is more informative for later modeling.

`daily_social_media_hours` shows the clearest separation between classes. Healthy teens average about 2.6 hours per day; At Risk averages about 7.1 hours (see the group means table and boxplot). Platform choice shows smaller differences than hours of use.

The lifestyle numeric correlation matrix does not show strong redundancy among the nine predictors at this stage.

Stress, anxiety, addiction, and the derived risk score remain excluded from modeling. Verified on all rows: risk score equals the sum of the three psychological scales.


## Preprocessing

Lifestyle predictors only. We stratify an 80/20 train/test split on `digital_wellbeing_flag`, then fit encoding and scaling on the training rows alone.

Engineered feature (decision, not a validated finding):

`high_screen_before_bed = 1 if screen_time_before_sleep > 2.0 else 0`

Categoricals (`gender`, `platform_usage`, `social_interaction_level`) are one-hot encoded with the first level dropped. Numeric lifestyle columns are standardized. No model training in this section.


In [ ]:
import features
import preprocessing
from config import LIFESTYLE_FEATURES, SEED, TARGET, TARGET_CLASSES, TEST_SIZE, EXCLUDED_COLUMNS
from io_utils import save_joblib
from paths import models_dir

# Fail fast with Colab setup hints if src/ was not cloned or wired correctly.
diag = features.verify_colab_src_setup()
print("Colab/src diagnostics OK")
print("  features loaded from:", diag["module_file"])
print("  src on sys.path:", diag["src_on_sys_path_count"] > 0)
print("preprocessing loaded from:", preprocessing.__file__)
print("models_dir:", models_dir())


In [ ]:
X, y = preprocessing.make_xy(df)

assert list(X.columns) == LIFESTYLE_FEATURES
assert y.name == TARGET
for col in EXCLUDED_COLUMNS:
    assert col not in X.columns, f"Excluded column leaked into X: {col}"

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Lifestyle columns:", list(X.columns))


In [ ]:
X_train, X_test, y_train, y_test = preprocessing.stratified_split(X, y)

train_idx = X_train.index
test_idx = X_test.index

split_counts = pd.DataFrame(
    {
        "train": y_train.value_counts().reindex(list(TARGET_CLASSES)),
        "test": y_test.value_counts().reindex(list(TARGET_CLASSES)),
    }
)
split_pct = pd.DataFrame(
    {
        "train_%": (y_train.value_counts(normalize=True) * 100)
        .reindex(list(TARGET_CLASSES))
        .round(1),
        "test_%": (y_test.value_counts(normalize=True) * 100)
        .reindex(list(TARGET_CLASSES))
        .round(1),
    }
)

print(f"train={len(X_train)} test={len(X_test)} (TEST_SIZE={TEST_SIZE}, SEED={SEED})")
display(split_counts)
display(split_pct)


In [ ]:
pipe = preprocessing.build_preprocessing_pipeline()
print(pipe)


In [ ]:
pipe, X_train_processed, X_test_processed, feature_names = (
    preprocessing.fit_transform_train_test(X_train, X_test, pipe)
)

print("X_train_processed:", getattr(X_train_processed, "shape", None))
print("X_test_processed:", getattr(X_test_processed, "shape", None))
print("n_features:", len(feature_names))
print("feature_names_out:")
for name in feature_names:
    print(" -", name)


In [ ]:
# Smoke checks: lifestyle-only, train-only fit, shapes, stratification
assert X_train_processed.shape == (960, preprocessing.EXPECTED_PROCESSED_N_FEATURES)
assert X_test_processed.shape == (240, preprocessing.EXPECTED_PROCESSED_N_FEATURES)
assert len(feature_names) == preprocessing.EXPECTED_PROCESSED_N_FEATURES

expected_train = {"Moderate": 594, "Healthy": 245, "At Risk": 121}
expected_test = {"Moderate": 149, "Healthy": 61, "At Risk": 30}
for label in TARGET_CLASSES:
    assert int(y_train.value_counts()[label]) == expected_train[label]
    assert int(y_test.value_counts()[label]) == expected_test[label]

# Engineered column present after engineer step (train copy only for inspection)
X_train_eng = features.add_engineered_features(X_train)
assert features.ENGINEERED_FEATURE in X_train_eng.columns
assert set(X_train_eng[features.ENGINEERED_FEATURE].unique()).issubset({0, 1})

print("Phase 3 smoke tests passed.")
print("Preprocessor was fit on train only; test was transformed with that fit.")


In [ ]:
artifact = preprocessing.build_split_artifact(
    train_idx=train_idx,
    test_idx=test_idx,
    X_train_raw=X_train,
    X_test_raw=X_test,
    y_train=y_train,
    y_test=y_test,
    X_train_processed=X_train_processed,
    X_test_processed=X_test_processed,
    feature_names_out=feature_names,
)

pipe_path = models_dir() / preprocessing.PREPROCESS_PIPELINE_FILENAME
split_path = models_dir() / preprocessing.TRAIN_TEST_SPLIT_FILENAME

save_joblib(pipe_path, pipe)
save_joblib(split_path, artifact)

print("Saved:", pipe_path)
print("Saved:", split_path)
assert pipe_path.is_file() and pipe_path.stat().st_size > 0
assert split_path.is_file() and split_path.stat().st_size > 0


## Preprocessing summary

Split: stratified 80/20 on `digital_wellbeing_flag`, `random_state=42` (960 train, 240 test). Class shares match the full data within rounding.

Encoding: one-hot for `gender`, `platform_usage`, and `social_interaction_level` (`drop="first"`). Numeric lifestyle columns use `StandardScaler`. The engineered binary `high_screen_before_bed` is passed through without scaling.

Formula: `high_screen_before_bed = 1 if screen_time_before_sleep > 2.0 else 0`. Threshold is fixed by decision, not learned from labels.

Processed width is 14 columns. Artifacts: `outputs/models/preprocess_pipeline.joblib` and `outputs/models/train_test_split.joblib`.

No classifier was trained here. Modeling is Phase 4.


## Export run summary

After a full run, this section writes one ZIP you can download and keep with your project notes:

`outputs/run_summary.zip`

Contents: `run_log.md` (short notes), `manifest.json` (exact numbers), and small CSVs under `tables/`. Edit `RUN_NOTES` below if you want to record anything worth revisiting.


In [ ]:
RUN_NOTES = """
# optional — edit before export
# e.g. platform crosstab feels noisy; might drop from report
"""

from run_summary import write_run_summary
from paths import run_summary_zip_path

zip_path = write_run_summary(
    df=df,
    y_train=y_train,
    y_test=y_test,
    feature_names=feature_names,
    notes=RUN_NOTES.strip(),
)
print("Wrote:", zip_path)
print("Expected path:", run_summary_zip_path())
assert zip_path.is_file() and zip_path.stat().st_size > 0


In [ ]:
try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Not on Colab — zip is at", zip_path)
